In [9]:
# ========================
# 1. Imports
# ========================
import numpy as np
import pandas as pd
import random, re
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, GlobalMaxPooling1D, Dense, Dropout
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

# ========================
# 2. Synthetic Dataset
# ========================
disease_symptoms = {
    "Influenza": ["fever", "sore throat", "body ache", "chills", "fatigue"],
    "Diabetes": ["frequent urination", "increased thirst", "blurred vision", "fatigue"]
}

data = []
for disease, symptoms in disease_symptoms.items():
    for _ in range(50):
        selected = random.sample(symptoms, random.randint(2, len(symptoms)))
        text = "Patient reports " + " and ".join(selected)
        data.append([text, disease])

df = pd.DataFrame(data, columns=["Symptoms", "Disease"])
print("Dataset size:", df.shape)
df.head()

# ========================
# 3. Preprocessing
# ========================
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords
stop_words = set(stopwords.words('english'))

def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    tokens = [w for w in text.split() if w not in stop_words]
    return " ".join(tokens)

df["Clean_Text"] = df["Symptoms"].apply(clean_text)

# ========================
# 4. Tokenization & Encoding
# ========================
tokenizer = Tokenizer(num_words=5000, oov_token="<OOV>")
tokenizer.fit_on_texts(df["Clean_Text"])
X = tokenizer.texts_to_sequences(df["Clean_Text"])
X = pad_sequences(X, maxlen=10, padding='post')

le = LabelEncoder()
y = le.fit_transform(df["Disease"])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# ========================
# 5. Build & Train ANN
# ========================
model = Sequential([
    Embedding(5000, 32, input_length=10),
    GlobalMaxPooling1D(),
    Dense(32, activation='relu'),
    Dropout(0.2),
    Dense(len(le.classes_), activation='softmax')
])
model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

history = model.fit(X_train, y_train, epochs=3, validation_data=(X_test, y_test))

# ========================
# 6. Predictions
# ========================
pred_probs = model.predict(X_test[:5])
pred_classes = np.argmax(pred_probs, axis=1)
pred_labels = le.inverse_transform(pred_classes)
print("Predicted labels for first 5 samples:", pred_labels)

# ========================
# 7. Predict function
# ========================
def predict_disease(text):
    seq = tokenizer.texts_to_sequences([clean_text(text)])
    seq = pad_sequences(seq, maxlen=10, padding='post')
    pred = model.predict(seq)
    return le.classes_[np.argmax(pred)]

example = "Patient reports fever and body ache"
print("Predicted Disease:", predict_disease(example))


Dataset size: (100, 2)
Epoch 1/3


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\watve\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
C:\Users\watve\Prathamesh\Lib\site-packages\keras\src\layers\core\embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


3/3 ━━━━━━━━━━━━━━━━━━━━ 2s 175ms/step - accuracy: 0.5000 - loss: 0.6934 - val_accuracy: 1.0000 - val_loss: 0.6704
Epoch 2/3
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 76ms/step - accuracy: 0.8500 - loss: 0.6675 - val_accuracy: 1.0000 - val_loss: 0.6491
Epoch 3/3
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step - accuracy: 0.9875 - loss: 0.6397 - val_accuracy: 1.0000 - val_loss: 0.6274
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 83ms/step
Predicted labels for first 5 samples: ['Influenza' 'Influenza' 'Influenza' 'Influenza' 'Diabetes']
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 96ms/step
Predicted Disease: Influenza
